# Team 7: Spark-first groundwater preprocessing

This notebook runs the real preprocessing in PySpark. pandas is not used to generate its outputs.
[Open in Colab](https://colab.research.google.com/github/dhruvvvgg/bde-team7/blob/spark-first-preprocessing/notebooks/preprocessing_spark.ipynb)


In [ ]:
from pathlib import Path
import subprocess, sys, os, json
repo = (Path.cwd() if (Path.cwd() / "scripts/run_spark.py").exists()
        else Path("/content/bde-team7") if Path("/content").exists() else Path.cwd() / "bde-team7")
if not repo.exists():
    subprocess.run(["git", "clone", "--branch", "spark-first-preprocessing", "https://github.com/dhruvvvgg/bde-team7.git", str(repo)], check=True)
if not (repo / "scripts/run_spark.py").exists():
    raise RuntimeError("Existing checkout lacks the migration; check out spark-first-preprocessing first.")
os.chdir(repo)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-spark.txt"], check=True)
subprocess.run(["java", "-version"], check=True)


## Execute preprocessing

Spark reads the source CSV, reshapes, cleans, joins rain, computes history/features, assigns labels and splits, and writes Parquet. The sampled CHIRPS tables are already committed. Expect several minutes on a local runtime.


In [ ]:
env = os.environ.copy()
env.setdefault("SPARK_LOCAL_IP", "127.0.0.1")
subprocess.run([sys.executable, "scripts/run_spark.py", "--master", "local[2]"], env=env, check=True)
manifest = json.loads(Path("processed/spark/run_manifest.json").read_text())
print(json.dumps(manifest, indent=2))
assert manifest["engine"] == "pyspark"
assert manifest["main_rows"] == 253828
assert manifest["main_columns"] == 84


## Explore the Spark output

These are grouped summaries after genuine Spark preprocessing, rather than the only Spark operations in the project.


In [ ]:
os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")
from pyspark.sql import SparkSession, functions as F
spark = SparkSession.builder.master("local[2]").appName("team7-explore").getOrCreate()
df = spark.read.parquet("processed/spark/gw_features_by_state")
df.printSchema()
df.groupBy("season").agg(F.avg("gwl_m_bgl").alias("mean_depth_m")).show()
df.groupBy("stress_category").count().show()
splits = spark.read.parquet("processed/spark/splits.parquet")
splits.filter("eval_eligible").groupBy("split_chrono").count().show()
spark.stop()
print(Path("processed/spark/execution_plan.txt").read_text()[:12000])


## Tests

Independent tests check temporal boundaries, missing rounds and label rules. Full pandas parity is optional validation outside the processing pipeline.


In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q", "tests/test_spark_pipeline.py"], env=env, check=True)


## Explain the architecture

Spark owns CSV ingestion, reshape, cleaning, rainfall joins, temporal feature windows, labels, splits and Parquet writes. Rasterio separately acquired the sampled rainfall inputs. Small executor-side UDFs implement specialised statistics. No warehouse is required. `local[2]` means one host, not a multi-node cluster. See `docs/spark_preprocessing.md` for scientific limitations and the intentional extrema correction.
